In [2]:
import os 
import glob

import numpy as np 
from shutil import rmtree 

In [6]:
def clean():
    for name in ("my-experiment", "ioh_data"):
        for path in glob.glob(f"{name}*"):
            if os.path.isfile(path):
                os.remove(path)
            if os.path.isdir(path):
                rmtree(path, ignore_errors=True)


def ls(p="./"):
    for obj in os.listdir(os.path.normpath(p)):
        print(obj)


def cat(f):
    with open(os.path.normpath(f)) as h:
        print(h.read())


clean()

In [7]:
import ioh 
%pip show ioh 

/home/sething2002/2025_S2/COMP_SCI_3316_Self/ioh_experiment/.venv/bin/python: No module named pip
Note: you may need to restart the kernel to use updated packages.


In [10]:
# a list of problems can be accessed via the base classes 
real_problems: dict[int, str] = ioh.problem.RealSingleObjective.problems
print(real_problems)

{1: 'Sphere', 2: 'Ellipsoid', 3: 'Rastrigin', 4: 'BuecheRastrigin', 5: 'LinearSlope', 6: 'AttractiveSector', 7: 'StepEllipsoid', 8: 'Rosenbrock', 9: 'RosenbrockRotated', 10: 'EllipsoidRotated', 11: 'Discus', 12: 'BentCigar', 13: 'SharpRidge', 14: 'DifferentPowers', 15: 'RastriginRotated', 16: 'Weierstrass', 17: 'Schaffers10', 18: 'Schaffers1000', 19: 'GriewankRosenbrock', 20: 'Schwefel', 21: 'Gallagher101', 22: 'Gallagher21', 23: 'Katsuura', 24: 'LunacekBiRastrigin', 30: 'UniformStarDiscrepancy10', 31: 'UniformStarDiscrepancy25', 32: 'UniformStarDiscrepancy50', 33: 'UniformStarDiscrepancy100', 34: 'UniformStarDiscrepancy150', 35: 'UniformStarDiscrepancy200', 36: 'UniformStarDiscrepancy250', 37: 'UniformStarDiscrepancy500', 38: 'UniformStarDiscrepancy750', 39: 'UniformStarDiscrepancy1000', 40: 'SobolStarDiscrepancy10', 41: 'SobolStarDiscrepancy25', 42: 'SobolStarDiscrepancy50', 43: 'SobolStarDiscrepancy100', 44: 'SobolStarDiscrepancy150', 45: 'SobolStarDiscrepancy200', 46: 'SobolStarDis

In [14]:
# in order to instantiate a problem instance, we can do the following: 

function_id = real_problems[1]


problem = ioh.get_problem(
    fid = function_id, 
    instance=1,
    dimension = 10, 
    problem_class = ioh.ProblemClass.REAL  # pyright: ignore[reportCallIssue]
)

problem 

<RealSingleObjectiveProblem 1. Sphere (iid=1 dim=10)>

In [15]:
# the problem class includes information about the problem, which can be retrieved via the meta data accessor 
problem.meta_data.name 

'Sphere'

In [16]:
# the current state of the problem, e.g., the number of evaluations, best seen points, etc. are stored in the problem state 
problem.state 

<State evaluations: 0 final_target_found: false current_best: <Solution x: [nan, nan, nan, nan, nan, nan, nan, nan, nan, nan] y: inf>>

In [17]:
# every problem has a simple box-bound assocaited 
problem.bounds 

<BoxConstraint lb: [[-5, -5, -5, -5, -5, -5, -5, -5, -5, -5]] ub: [[5, 5, 5, 5, 5, 5, 5, 5, 5, 5]]>

In [19]:
# we can access the constraint information of the problem 

x0 = np.random.uniform(problem.bounds.lb, problem.bounds.ub)


# evaluation happens like a 'normal' objective function would 
problem(x0)

print("-----------------------")

# whenever the problem is evaluated, the state changes 
problem.state 

-----------------------


<State evaluations: 2 final_target_found: false current_best: <Solution x: [-4.483097444038037, 0.6170430248806369, 2.565784877488814, -4.975482568766292, -0.9708010761166701, -2.1611043305620283, 2.2355732137719384, -1.2524037451611472, -3.3518680406073678, 2.077601529411397] y: 209.86006196152414>>

In [21]:
# additionally, it is possible to evaluate a list of points 

n_points: int = 5 
X = np.random.uniform(problem.bounds.lb, problem.bounds.ub, size=(n_points, problem.meta_data.n_variables))

In [22]:
# if we want to perform multiple runs with the same objective function, after every run, the problem has to be reset 

def run_experiment(problem, algorithm, n_runs=5):
    for run in range(n_runs):


        # run the algorithm on the problem 
        algorithm(problem)

        # print the best found for this run 
        print(f"run: {run + 1} - best found: {problem.state.current_best.y: .3f}")


        # reset the problem 
        problem.reset()

In [23]:
class RandomSearch:
    def __init__(self, n: int, length: float = 0.0):
        self.n: int = n 
        self.length: float = length 
    


    def __call__(self, problem: ioh.problem.RealSingleObjective) -> None:
        # evaluate the problem n times with a randomly generated solution


        for _ in range(self.n):
            # we can use the problems bound accessor to get information about the problem bounds 
            x = np.random.uniform(problem.bounds.lb, problem.bounds.ub)
            self.length = float(np.linalg.norm(x))


            problem(x)
            

In [24]:
# using the random search algorithm, we can then run a simple experiment 
run_experiment(problem, RandomSearch(10))

run: 1 - best found:  122.081
run: 2 - best found:  117.299
run: 3 - best found:  145.284
run: 4 - best found:  130.931
run: 5 - best found:  139.887


In [25]:
def styblinsky_tang(x: np.ndarray) -> float:
    return np.sum(np.power(x, 4) - (16 * np.power(x, 2)) + (5 * x)) / 2




styblinsky_tang(np.array([-2.903534]*10)) # global minima

np.float64(-391.661657037714)

In [ ]:
# we can create an instance of this problem wrapped in ioh 

